# GEAS3.5 실시간 결측값 처리 실험: 연속형 컬럼 Kalman Filter

이 노트북은 `GEAS3.5/source` 본 코드에 넣기 전에 `data_silla_enc` 41개 컬럼의 결측값 처리 방식을 시험하기 위한 precode입니다.

- 이상치 처리가 아니라 결측값 처리만 다룹니다.
- 연속형 센서/상태 컬럼에만 1차원 Kalman Filter 기반 대체를 적용합니다.
- 이산형 장치 상태값과 식별자/시간 컬럼은 우선 대체하지 않습니다.
- Kalman 대체값은 원본 컬럼을 덮어쓰지 않고 `<column>_kf`, `<column>_imputed_flag`, `<column>_quality` 컬럼으로 따로 만듭니다.

In [1]:
from __future__ import annotations

import json
import os
import re
import sys
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import pymysql

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "precode" else NOTEBOOK_DIR
GEAS35_SOURCE = REPO_ROOT / "GEAS3.5" / "source"
GEAS35_INNER = GEAS35_SOURCE / "inner_layer"
COLUMN_INFO_PATH = REPO_ROOT / "code_review" / "db_structure_inspection" / "column_info.csv"

for path in (GEAS35_SOURCE, GEAS35_INNER):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from core.config import DbConfig

DB_ENV_DEFAULTS = {
    "DB_HOST": "211.195.9.227",
    "DB_PORT": "3306",
    "DB_USER": "root",
    "DB_NAME": "farmstom",
    "FARM_SN": "97",
}

for key, value in DB_ENV_DEFAULTS.items():
    os.environ.setdefault(key, value)

def load_saved_db_password() -> str:
    inspector_path = REPO_ROOT / "code_review" / "db_structure_inspection" / "db_structure_inspector.ipynb"
    if not inspector_path.exists():
        return ""
    try:
        notebook = json.loads(inspector_path.read_text(encoding="utf-8"))
    except Exception:
        return ""
    for cell in notebook.get("cells", []):
        source = "".join(cell.get("source", []))
        match = re.search(r"password\s*=\s*['\"]([^'\"]+)['\"]", source)
        if match:
            return match.group(1)
    return ""

password_source = "environment"
if not os.getenv("DB_PASSWORD"):
    saved_password = load_saved_db_password()
    if saved_password:
        os.environ["DB_PASSWORD"] = saved_password
        password_source = "db_structure_inspector.ipynb"
    else:
        os.environ["DB_PASSWORD"] = getpass("DB_PASSWORD (not saved in notebook): ")
        password_source = "manual input"

db_cfg_preview = DbConfig.from_env()

column_info = pd.read_csv(COLUMN_INFO_PATH, encoding="utf-8")
ALL_DB_COLUMNS = column_info["컬럼명"].astype(str).tolist()

print(f"repo root: {REPO_ROOT}")
print(f"GEAS3.5 source: {GEAS35_SOURCE}")
print(f"data_silla_enc column count from column_info.csv: {len(ALL_DB_COLUMNS)}")
print(
    "DB config: "
    f"host={db_cfg_preview.host}, port={db_cfg_preview.port}, "
    f"user={db_cfg_preview.user}, database={db_cfg_preview.name}, "
    f"farm_sn={db_cfg_preview.farm_sn}, "
    f"password={'set' if bool(db_cfg_preview.password) else 'missing'} ({password_source})"
)

repo root: c:\Users\10-64\GEAS
GEAS3.5 source: c:\Users\10-64\GEAS\GEAS3.5\source
data_silla_enc column count from column_info.csv: 41
DB config: host=211.195.9.227, port=3306, user=root, database=farmstom, farm_sn=97, password=set (db_structure_inspector.ipynb)


## 컬럼 그룹 정의

`CONTINUOUS_KALMAN_COLUMNS`에 포함된 컬럼만 Kalman Filter 적용 대상입니다. `out_winddirec`는 숫자형이지만 0도와 360도가 이어지는 원형 변수라 단순 1차원 Kalman Filter 대상에서 제외했습니다.

In [2]:
IDENTIFIER_TIME_COLUMNS = [
    "idx",
    "iot_data_idx",
    "reg_date",
]

CONTINUOUS_KALMAN_COLUMNS = [
    "in_medium_temp1",
    "in_medium_temp2",
    "in_temp",
    "in_temp2",
    "in_water_hot",
    "in_water_cold",
    "in_hum",
    "in_hum2",
    "in_medium_hum1",
    "in_medium_hum2",
    "in_co2",
    "in_co2_2",
    "in_medium_ec1",
    "in_medium_ec2",
    "out_temp",
    "out_hum",
    "out_windsp",
    "out_light",
    "out_light_sum",
    "out_rainfall",
    "out_airpress",
    "cont_skyl_vol",
    "cont_skyr_vol",
    "cont_cur_vol",
    "cont_kwcur_vol",
    "cont_3way1_vol",
    "cont_3way2_vol",
]

DISCRETE_OR_HELD_COLUMNS = [
    "out_rain",
    "etc_blackout",
    "etc_plc_abnorm",
    "etc_plc_norm",
    "cont_heater_run",
    "cont_cooler_run",
    "cont_co2_run",
    "cont_pump1_run",
    "cont_pump2_run",
    "cont_fan_run",
]

CIRCULAR_CONTINUOUS_SKIP_COLUMNS = ["out_winddirec"]

classified = set(IDENTIFIER_TIME_COLUMNS + CONTINUOUS_KALMAN_COLUMNS + DISCRETE_OR_HELD_COLUMNS + CIRCULAR_CONTINUOUS_SKIP_COLUMNS)
unclassified = [col for col in ALL_DB_COLUMNS if col not in classified]
assert not unclassified, f"Unclassified columns: {unclassified}"

column_groups = pd.DataFrame(
    [
        {"column": col, "group": "identifier_or_time", "planned_missing_action": "no imputation"}
        for col in IDENTIFIER_TIME_COLUMNS
    ]
    + [
        {"column": col, "group": "continuous_kalman", "planned_missing_action": "Kalman filter candidate"}
        for col in CONTINUOUS_KALMAN_COLUMNS
    ]
    + [
        {"column": col, "group": "discrete_or_binary", "planned_missing_action": "leave as-is in this notebook"}
        for col in DISCRETE_OR_HELD_COLUMNS
    ]
    + [
        {"column": col, "group": "circular_continuous", "planned_missing_action": "skip simple 1D Kalman"}
        for col in CIRCULAR_CONTINUOUS_SKIP_COLUMNS
    ]
)

column_groups

,column,group,planned_missing_action
0,idx,identifier_or_time,no imputation
1,iot_data_idx,identifier_or_time,no imputation
2,reg_date,identifier_or_time,no imputation
3,in_medium_temp1,continuous_kalman,Kalman filter candidate
4,in_medium_temp2,continuous_kalman,Kalman filter candidate
5,in_temp,continuous_kalman,Kalman filter candidate
6,in_temp2,continuous_kalman,Kalman filter candidate
7,in_water_hot,continuous_kalman,Kalman filter candidate
8,in_water_cold,continuous_kalman,Kalman filter candidate
9,in_hum,continuous_kalman,Kalman filter candidate


## DB에서 41개 컬럼 읽기

`START`, `END`, `LIMIT` 값을 조정해서 실험 범위를 바꿉니다. 아래 기본 설정은 실제 분석 대상 전체 기간을 읽도록 `LIMIT = None`으로 둡니다.

In [3]:
TABLE_NAME = "data_silla_enc"
START = "2024-03-06 12:20:00"
END = "2026-06-25 00:00:00"
LIMIT = None


def quote_identifier(value: str) -> str:
    if not value or "\x00" in value:
        raise ValueError(f"Invalid SQL identifier: {value!r}")
    return "`" + value.replace("`", "``") + "`"


def fetch_data_silla_enc(
    *,
    start: str | None = START,
    end: str | None = END,
    limit: int | None = LIMIT,
    columns: list[str] = ALL_DB_COLUMNS,
) -> pd.DataFrame:
    cfg = DbConfig.from_env()
    select_cols = ", ".join(quote_identifier(col) for col in columns)
    where = ["iot_data_idx = %s"]
    params: list[object] = [cfg.farm_sn]

    if start is not None:
        where.append("reg_date >= %s")
        params.append(start)
    if end is not None:
        where.append("reg_date < %s")
        params.append(end)

    sql = f"""
        SELECT {select_cols}
        FROM {quote_identifier(TABLE_NAME)}
        WHERE {' AND '.join(where)}
        ORDER BY reg_date ASC
    """
    if limit is not None:
        sql += "\nLIMIT %s"
        params.append(int(limit))

    conn = pymysql.connect(
        host=cfg.host,
        port=cfg.port,
        user=cfg.user,
        password=cfg.password,
        database=cfg.name,
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            rows = cur.fetchall()
        return pd.DataFrame(rows, columns=columns)
    finally:
        conn.close()


df_raw = fetch_data_silla_enc()
print(f"loaded rows={len(df_raw):,}, columns={len(df_raw.columns):,}")
df_raw.head()

loaded rows=202,957, columns=41


,idx,iot_data_idx,reg_date,in_medium_temp1,in_medium_temp2,in_temp,in_temp2,in_water_hot,in_water_cold,in_hum,...,cont_cur_vol,cont_kwcur_vol,cont_heater_run,cont_cooler_run,cont_co2_run,cont_3way1_vol,cont_3way2_vol,cont_pump1_run,cont_pump2_run,cont_fan_run
0,13803,97,2024-03-06 12:20:00,8,0,32.2,33.7,35.99,33.03,36.2,...,0,100,0,0,0,0,0,0,0,1
1,13804,97,2024-03-06 12:30:00,8,0,32.2,33.7,35.99,33.03,36.2,...,8,100,0,0,0,0,0,0,0,1
2,13805,97,2024-03-06 12:40:00,8,0,31.92,23.7,35.99,33.03,24.8,...,10,100,0,0,0,0,0,0,0,1
3,13806,97,2024-03-06 12:50:00,8,0,29.73,30.07,35.99,33.03,23.87,...,10,100,0,0,0,0,0,0,0,1
4,13807,97,2024-03-06 13:00:00,8,0,26.69,26.98,35.99,33.03,27.53,...,10,100,0,0,0,0,0,0,0,1


## 연속형 Kalman 대상 컬럼 결측치 비율 표

아래 셀은 `df_raw`가 준비된 후 실행합니다. 빈 문자열, `NA`, `NULL` 같은 값도 결측으로 계산합니다.

In [4]:
MISSING_TOKENS = frozenset({"", "NA", "N/A", "NaN", "nan", "NULL", "null", "None", "none"})


def semantic_missing_mask(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.isna()
    object_cols = list(df.select_dtypes(include=["object", "string"]).columns)
    for col in object_cols:
        stripped = df[col].astype("string").str.strip()
        mask[col] = mask[col] | stripped.isin(MISSING_TOKENS)
    return mask


def continuous_missingness_table(df: pd.DataFrame) -> pd.DataFrame:
    present_cols = [col for col in CONTINUOUS_KALMAN_COLUMNS if col in df.columns]
    mask = semantic_missing_mask(df[present_cols])
    rows = len(df.index)
    out = pd.DataFrame(
        {
            "column": present_cols,
            "missing_count": [int(mask[col].sum()) for col in present_cols],
            "total_rows": rows,
            "missing_ratio": [float(mask[col].sum() / rows) if rows else np.nan for col in present_cols],
        }
    )
    out = out.sort_values(["missing_ratio", "missing_count", "column"], ascending=[False, False, True])
    return out.reset_index(drop=True)


continuous_missingness = continuous_missingness_table(df_raw)
continuous_missingness

,column,missing_count,total_rows,missing_ratio
0,cont_3way1_vol,0,202957,0.0
1,cont_3way2_vol,0,202957,0.0
2,cont_cur_vol,0,202957,0.0
3,cont_kwcur_vol,0,202957,0.0
4,cont_skyl_vol,0,202957,0.0
5,cont_skyr_vol,0,202957,0.0
6,in_co2,0,202957,0.0
7,in_co2_2,0,202957,0.0
8,in_hum,0,202957,0.0
9,in_hum2,0,202957,0.0


## 1차원 Kalman Filter 결측 대체 함수

여기서는 엣지 적용을 염두에 두고 외부 Kalman 라이브러리 없이 random-walk 형태의 가벼운 1차원 Kalman Filter를 사용합니다. 현재값이 결측이면 예측값을 쓰되, `max_gap_steps`를 넘는 긴 결측 구간은 대체하지 않고 `missing_timeout`으로 남깁니다.

In [5]:
def estimate_measurement_var(values: np.ndarray, fallback: float = 1.0) -> float:
    finite = values[np.isfinite(values)]
    if finite.size < 3:
        return fallback
    diffs = np.diff(finite)
    var = float(np.nanvar(diffs))
    if not np.isfinite(var) or var <= 0:
        return fallback
    return max(var, 1e-6)


def kalman_impute_1d(
    series: pd.Series,
    *,
    process_var: float = 1e-3,
    measurement_var: float | None = None,
    initial_var: float = 1.0,
    max_gap_steps: int = 3,
) -> pd.DataFrame:
    y = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    n = len(y)
    out = np.full(n, np.nan, dtype=float)
    imputed = np.zeros(n, dtype=bool)
    quality = np.array(["missing_timeout"] * n, dtype=object)

    valid_idx = np.where(np.isfinite(y))[0]
    if valid_idx.size == 0:
        return pd.DataFrame({"value": out, "imputed_flag": imputed, "quality": quality})

    r = measurement_var if measurement_var is not None else estimate_measurement_var(y)
    q = max(float(process_var), 1e-12)
    x = float(y[valid_idx[0]])
    p = float(initial_var)
    last_observed_idx: int | None = None

    for i, obs in enumerate(y):
        x_pred = x
        p_pred = p + q

        if np.isfinite(obs):
            k = p_pred / (p_pred + r)
            x = x_pred + k * (float(obs) - x_pred)
            p = (1.0 - k) * p_pred
            out[i] = float(obs)
            quality[i] = "observed"
            last_observed_idx = i
        else:
            x = x_pred
            p = p_pred
            gap_steps = None if last_observed_idx is None else i - last_observed_idx
            if gap_steps is not None and gap_steps <= max_gap_steps:
                out[i] = x_pred
                imputed[i] = True
                quality[i] = "kalman_imputed"

    return pd.DataFrame({"value": out, "imputed_flag": imputed, "quality": quality})


def apply_kalman_to_continuous_columns(
    df: pd.DataFrame,
    *,
    max_gap_steps: int = 3,
    process_var: float = 1e-3,
) -> pd.DataFrame:
    out = df.copy()
    if "reg_date" in out.columns:
        out["reg_date"] = pd.to_datetime(out["reg_date"], errors="coerce")
        out = out.sort_values("reg_date").reset_index(drop=True)

    for col in CONTINUOUS_KALMAN_COLUMNS:
        if col not in out.columns:
            continue
        result = kalman_impute_1d(
            out[col],
            process_var=process_var,
            max_gap_steps=max_gap_steps,
        )
        out[f"{col}_kf"] = result["value"]
        out[f"{col}_imputed_flag"] = result["imputed_flag"]
        out[f"{col}_quality"] = result["quality"]

    return out


df_kf = apply_kalman_to_continuous_columns(df_raw, max_gap_steps=3)
print(f"Kalman result columns added: {len(df_kf.columns) - len(df_raw.columns):,}")
df_kf.head()

Kalman result columns added: 81


,idx,iot_data_idx,reg_date,in_medium_temp1,in_medium_temp2,in_temp,in_temp2,in_water_hot,in_water_cold,in_hum,...,cont_cur_vol_quality,cont_kwcur_vol_kf,cont_kwcur_vol_imputed_flag,cont_kwcur_vol_quality,cont_3way1_vol_kf,cont_3way1_vol_imputed_flag,cont_3way1_vol_quality,cont_3way2_vol_kf,cont_3way2_vol_imputed_flag,cont_3way2_vol_quality
0,13803,97,2024-03-06 12:20:00,8,0,32.2,33.7,35.99,33.03,36.2,...,observed,100.0,False,observed,0.0,False,observed,0.0,False,observed
1,13804,97,2024-03-06 12:30:00,8,0,32.2,33.7,35.99,33.03,36.2,...,observed,100.0,False,observed,0.0,False,observed,0.0,False,observed
2,13805,97,2024-03-06 12:40:00,8,0,31.92,23.7,35.99,33.03,24.8,...,observed,100.0,False,observed,0.0,False,observed,0.0,False,observed
3,13806,97,2024-03-06 12:50:00,8,0,29.73,30.07,35.99,33.03,23.87,...,observed,100.0,False,observed,0.0,False,observed,0.0,False,observed
4,13807,97,2024-03-06 13:00:00,8,0,26.69,26.98,35.99,33.03,27.53,...,observed,100.0,False,observed,0.0,False,observed,0.0,False,observed


## Kalman 적용 결과 요약

연속형 컬럼별로 실제 결측 중 몇 개가 Kalman으로 대체되었고, timeout 때문에 남은 결측이 몇 개인지 확인합니다.

In [6]:
def kalman_result_summary(df_kf: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in CONTINUOUS_KALMAN_COLUMNS:
        value_col = f"{col}_kf"
        flag_col = f"{col}_imputed_flag"
        quality_col = f"{col}_quality"
        if value_col not in df_kf.columns:
            continue
        quality_counts = df_kf[quality_col].value_counts(dropna=False).to_dict()
        rows.append(
            {
                "column": col,
                "observed_count": int(quality_counts.get("observed", 0)),
                "kalman_imputed_count": int(df_kf[flag_col].sum()),
                "missing_timeout_count": int(quality_counts.get("missing_timeout", 0)),
                "remaining_missing_after_kf": int(df_kf[value_col].isna().sum()),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["missing_timeout_count", "kalman_imputed_count", "column"],
        ascending=[False, False, True],
    ).reset_index(drop=True)


kalman_summary = kalman_result_summary(df_kf)
kalman_summary

,column,observed_count,kalman_imputed_count,missing_timeout_count,remaining_missing_after_kf
0,cont_3way1_vol,202957,0,0,0
1,cont_3way2_vol,202957,0,0,0
2,cont_cur_vol,202957,0,0,0
3,cont_kwcur_vol,202957,0,0,0
4,cont_skyl_vol,202957,0,0,0
5,cont_skyr_vol,202957,0,0,0
6,in_co2,202957,0,0,0
7,in_co2_2,202957,0,0,0
8,in_hum,202957,0,0,0
9,in_hum2,202957,0,0,0
